In [ ]:
# Make the `perovskite` package importable regardless of where Jupyter started.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "perovskite").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from perovskite.data import load_features_and_meta, make_split

# SOAP features + reduced metadata (target y)
X, df_meta = load_features_and_meta("soap")
y = df_meta["energy_above_hull"].values  # target vector
print("X:", X.shape, " y:", y.shape)
df_meta.head()

In [ ]:
# Train on the raw target -- no y-scaling (MinMax is a no-op for a RandomForest's R²).
X_train, X_test, y_train, y_test = make_split(X, y)
print("X_train Form:", X_train.shape)

In [38]:
# Berechne die Perzentile
quantiles = [0.50, 0.75, 0.90, 0.95, 0.99]
percentile_values = df_meta["energy_above_hull"].quantile(quantiles)

print("Quantitative Verteilung der Energy above Hull:")
print("-----------------------------------------------")
for q, val in zip(quantiles, percentile_values):
    print(f"{int(q*100):>5}% der Daten liegen unter: {val:.4f}")

skewness = df_meta["energy_above_hull"].skew()
print(f"\nSkewness (Schiefe): {skewness:.2f}")

Quantitative Verteilung der Energy above Hull:
-----------------------------------------------
   50% der Daten liegen unter: 0.2567
   75% der Daten liegen unter: 0.6293
   90% der Daten liegen unter: 1.3095
   95% der Daten liegen unter: 1.7571
   99% der Daten liegen unter: 2.7465

Skewness (Schiefe): 2.12


In [ ]:
import matplotlib.pyplot as plt

# Verteilung des (rohen) Targets im Trainingsset
plt.figure(figsize=(8, 4))
plt.hist(y_train, bins=50, color="skyblue", edgecolor="black", alpha=0.7)
plt.title("Verteilung: Energy above hull (Trainingsdaten)")
plt.xlabel("Energy above hull")
plt.ylabel("Häufigkeit")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest Regressor, trained on the raw target (no y-scaling).
rf_model = RandomForestRegressor(
    n_estimators=150,      # Anzahl der Bäume im Wald
    max_depth=10,          # Maximale Tiefe pro Baum
    min_samples_split=5,
    random_state=42,       # Macht die Ergebnisse reproduzierbar
    n_jobs=-1,             # Nutzen aller CPU-Kerne
)

print("Trainiere Random Forest Regressor (das kann bei vielen Spalten etwas dauern)...")
rf_model.fit(X_train, y_train)
print("Training abgeschlossen!")
# Evaluation erfolgt unten via evaluate_model_performance (Metriken in Original-Einheit).

In [ ]:
# Shared helper (was defined inline; now lives in perovskite/evaluation.py)
from perovskite.evaluation import evaluate_model_performance

In [ ]:
evaluate_model_performance(rf_model, X_test, y_test, title="Energy above hull (SOAP)")

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# 1. Das Basis-Modell definieren (ohne class_weight, da Regression)
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# 2. Den Suchraum (Grid) festlegen
param_grid = {
    "n_estimators": [50, 100],  # Anzahl der Bäume
    "max_depth": [5, 10, None],  # Tiefe der Bäume (None bedeutet unbegrenzt)
    "min_samples_split": [2, 5],  # Mindestanzahl an Datenpunkten für einen Split
}

# 3. Grid Search initialisieren
# scoring='neg_mean_squared_error' optimiert auf den geringsten quadratischen Fehler.
# Alternativ könntest du auch 'r2' nutzen.
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,  # 3-fache Kreuzvalidierung
    scoring="neg_mean_squared_error",  # <--- Regressions-Metrik
    verbose=2,  # Zeigt den Fortschritt live an
    n_jobs=-1,  # Nutzt alle CPU-Kerne parallel
)

# 4. Suche starten (auf dem rohen Target)
print("Starte Grid Search für Regression...")
grid_search.fit(X_train, y_train)

# 5. Bestes Ergebnis anzeigen
print("\n=== GRID SEARCH BEENDET ===")
# Wir nehmen das Negative des Scores und ziehen die Wurzel, um den echten RMSE anzuzeigen
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Bester RMSE-Score (ca. Abweichung): {best_rmse:.2f}")
print("Beste Parameter-Kombination:")
print(grid_search.best_params_)

# Das beste Modell direkt für Vorhersagen nutzen
best_rf_regressor = grid_search.best_estimator_

In [ ]:
evaluate_model_performance(best_rf_regressor, X_test, y_test, title="Energy above hull (SOAP, tuned)")

Gute Gesamtabdeckung (R 
2
 =0.84): Das Modell ist stark. Es kann rund 84 % der Varianz der Energy above Hull auf komplett unbekannten Testdaten korrekt erklären. Für ein komplexes physikalisches Regressionsmodell ist das ein sehr solider Wert.
Präzise Vorhersage im Alltag (MAE = 0.12): Im Durchschnitt weichen die Vorhersagen des Modells nur um 0.12 Einheiten vom tatsächlichen Wert ab. Das zeigt, dass der Großteil der Strukturen sehr präzise geschätzt wird.
Einfluss von Extremwerten (RMSE vs. MAE): Der RMSE ist mit 0.25 doppelt so hoch wie der MAE. Das ist die mathematische Bestätigung für deine Beobachtung, dass vereinzelt sehr große Energiewerte im Datensatz existieren. Da der RMSE große Abweichungen quadratisch bestraft, wird er durch diese Ausreißer nach oben gezogen.
Die Schwachstelle (Max Error = 4.54): Der maximale Fehler zeigt, dass das Modell bei mindestens einer Struktur (sehr wahrscheinlich eine der vereinzelten, extrem instabilen Phasen) um 4.54 Einheiten daneben lag.

# ==========================================================

## Try Ewald Sum Matrix

In [ ]:
# Ewald-sum-matrix features + reduced metadata
X, df_meta = load_features_and_meta("ewald")
y = df_meta["energy_above_hull"].values  # target vector
print("X:", X.shape, " y:", y.shape)
df_meta.head()

In [ ]:
X_train, X_test, y_train, y_test = make_split(X, y)
print("X_train Form:", X_train.shape)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest Regressor on Ewald features, trained on the raw target.
rf_ew_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
)

print("Trainiere Random Forest Regressor (Ewald)...")
rf_ew_model.fit(X_train, y_train)
print("Training abgeschlossen!")

In [ ]:
evaluate_model_performance(rf_ew_model, X_test, y_test, title="Energy above hull (Ewald)")

## Grid search with Ewald Matrices

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# 1. Das Basis-Modell definieren (ohne class_weight, da Regression)
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# 2. Den Suchraum (Grid) festlegen
param_grid = {
    "n_estimators": [50, 100, 200],  # Anzahl der Bäume
    "max_depth": [5, 10, None],  # Tiefe der Bäume (None bedeutet unbegrenzt)
    "min_samples_split": [4, 6, 8],  # Mindestanzahl an Datenpunkten für einen Split
}

# 3. Grid Search initialisieren
# scoring='neg_mean_squared_error' optimiert auf den geringsten quadratischen Fehler.
# Alternativ könntest du auch 'r2' nutzen.
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,  # 3-fache Kreuzvalidierung
    scoring="neg_mean_squared_error",  # <--- Regressions-Metrik
    verbose=2,  # Zeigt den Fortschritt live an
    n_jobs=-1,  # Nutzt alle CPU-Kerne parallel
)

# 4. Suche starten (auf dem rohen Target)
print("Starte Grid Search für Regression...")
grid_search.fit(X_train, y_train)

# 5. Bestes Ergebnis anzeigen
print("\n=== GRID SEARCH BEENDET ===")
# Wir nehmen das Negative des Scores und ziehen die Wurzel, um den echten RMSE anzuzeigen
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Bester RMSE-Score (ca. Abweichung): {best_rmse:.2f}")
print("Beste Parameter-Kombination:")
print(grid_search.best_params_)

# Das beste Modell direkt für Vorhersagen nutzen
best_rf_ew_regressor = grid_search.best_estimator_

In [ ]:
evaluate_model_performance(best_rf_ew_regressor, X_test, y_test, title="Energy above hull (Ewald, tuned)")